In [3]:
import os
import joblib
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Load dataset
data = fetch_california_housing()
X, y = data.data, data.target
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Load sklearn model
model = joblib.load("models/sklearn_model.joblib")
coef = model.coef_
intercept = model.intercept_

# Save unquantized parameters
joblib.dump({'weights': coef, 'bias': intercept}, "models/unquant_params.joblib")

# Manual quantization
def quantize(arr):
    min_val = arr.min()
    max_val = arr.max()
    scale = 255.0 / (max_val - min_val) if max_val != min_val else 1.0
    q_arr = ((arr - min_val) * scale).astype(np.uint8)
    return q_arr, min_val, scale

def dequantize(q_arr, min_val, scale):
    return q_arr.astype(np.float32) / scale + min_val

w_q, w_min, w_scale = quantize(coef)
b_q, b_min, b_scale = quantize(np.array([intercept]))

# Save quantized
joblib.dump({
    "weights_uint8": w_q,
    "bias_uint8": b_q,
    "w_min": w_min, "w_scale": w_scale,
    "b_min": b_min, "b_scale": b_scale
}, "models/quant_params.joblib")

# Dequantize
w_dq = dequantize(w_q, w_min, w_scale)
b_dq = dequantize(b_q, b_min, b_scale)[0]

# Rebuild PyTorch model with quantized params
class TorchModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)

model_q = TorchModel(X.shape[1])
with torch.no_grad():
    model_q.linear.weight.copy_(torch.tensor(w_dq, dtype=torch.float32).unsqueeze(0))
    model_q.linear.bias.copy_(torch.tensor(b_dq, dtype=torch.float32))

# Evaluate
X_tensor = torch.tensor(X_test, dtype=torch.float32)
model_q.eval()
with torch.no_grad():
    y_pred = model_q(X_tensor).squeeze().numpy()

r2 = r2_score(y_test, y_pred)
#print(f"R² Score (Quantized PyTorch): {r2:.4f}")
#print(f"Model Size (unquantized): {os.path.getsize('models/unquant_params.joblib') / 1024:.2f} KB")
#print(f"Model Size (quantized):   {os.path.getsize('models/quant_params.joblib') / 1024:.2f} KB")

print("\n" + "="*40)
print(f"{'Metric':<20}{'Sklearn Model':<20}{'Quantized Model'}")
print(f"{'-'*60}")
print(f"{'R² Score':<20}{r2_score(y_test, joblib.load('models/sklearn_model.joblib').predict(X_test)):<20.4f}{r2:.4f}")
print(f"{'Model Size KB':<20}{os.path.getsize('models/unquant_params.joblib') / 1024:<20.2f}{os.path.getsize('models/quant_params.joblib') / 1024:.2f}")
print("="*40 + "\n")



Metric              Sklearn Model       Quantized Model
------------------------------------------------------------
R² Score            0.5758              -46.6831
Model Size KB       0.40                0.51

